# MongoDB Fundamentals

## Notebook Objectives

This notebook introduces the fundamentals of MongoDB using both **PyMongo** and **MongoDB Shell (mongosh)**.

Throughout this notebook, every MongoDB operation will be demonstrated using:

- **PyMongo** – The official Python driver for MongoDB.
- **MongoDB Shell (mongosh)** – The command-line interface used to interact with MongoDB.

By the end of this notebook, you will be able to:

- Connect to a MongoDB server.
- Create databases and collections.
- Insert, retrieve, update, and delete documents.
- Use MongoDB query operators.
- Perform aggregation pipeline operations.

> **Note:** For operations that modify the database (Insert, Update, Delete), the **mongosh** commands are provided as commented code to prevent duplicate execution. You can copy and execute them directly in the MongoDB Shell whenever required.

# Import Required Libraries

The following Python libraries will be used throughout this notebook.

| Library | Purpose |
|----------|----------|
| `pymongo` | Connect Python applications with MongoDB |
| `pandas` | Data manipulation and analysis |
| `numpy` | Numerical computations |
| `matplotlib` | Data visualization |
| `seaborn` | Statistical visualization |

In [57]:
# =====================================
# Import Required Libraries
# =====================================

from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

print("All required libraries imported successfully.")

All required libraries imported successfully.


# Connect to MongoDB Server

Before performing any database operations, we must establish a connection with the MongoDB server.

The default connection URL is:

```text
mongodb://localhost:27017
```

where

- **localhost** → MongoDB Server
- **27017** → Default MongoDB Port

After creating the client, we will verify that the MongoDB server is running successfully.

In [3]:
# =====================================
# PyMongo Command
# =====================================

client = MongoClient("mongodb://localhost:27017")

try:
    client.admin.command("ping")
    print("Successfully connected to MongoDB Server.")
except Exception as e:
    print("Connection Failed")
    print(e)

Successfully connected to MongoDB Server.


In [4]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.adminCommand({ ping: 1 })"

{ ok: 1 }


# Display Available Databases

MongoDB stores multiple databases on a single server.

The following commands display all currently available databases.

> **Note:** MongoDB displays only those databases that contain at least one collection with data.

In [9]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [10]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Database

A **database** is a logical container that stores one or more **collections**.

Just as a folder on your computer can contain multiple files, a MongoDB database can contain multiple collections.

In this notebook, we will create a database named **ecommerce**.

> **Important:**  
> In MongoDB, simply selecting a database **does not create it**. The database is physically created **only after a document is inserted into one of its collections**.

For now, we will simply select the database. It will appear in MongoDB only after data is inserted in later sections.

In [11]:
# =====================================
# PyMongo Command
# =====================================

db = client["ecommerce"]

db

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce')

In [12]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce"

switched to db ecommerce


# Understanding the Database Object

The variable **`db`** is a **Database object** returned by PyMongo.

It represents the selected MongoDB database and allows us to perform operations such as:

- Creating collections
- Listing collections
- Inserting documents
- Querying data
- Updating documents
- Deleting documents

Since no documents have been inserted yet, the **ecommerce** database does not physically exist on the MongoDB server.

You can verify this by listing all available databases.

In [13]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'local']

In [14]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin    40.00 KiB
config  108.00 KiB
local    64.00 KiB


# Collection

A **collection** is a group of related documents stored within a MongoDB database.

It is similar to a **table** in a relational database (RDBMS), but unlike a table:

- Collections do **not** require a fixed schema.
- Documents in the same collection can have different fields.
- MongoDB automatically creates a collection when the first document is inserted into it.

For better understanding, consider the following comparison:

| Relational Database | MongoDB |
|---------------------|----------|
| Database | Database |
| Table | Collection |
| Row | Document |
| Column | Field |

In this notebook, we will use a collection named **customers** to store customer information for our e-commerce application.

> **Important:**  
> Like databases, selecting a collection does **not** physically create it. The collection is created automatically when the first document is inserted.

In [15]:
# =====================================
# PyMongo Command
# =====================================

customers = db["customers"]

customers

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'ecommerce'), 'customers')

In [16]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.getCollection('customers')"

switched to db ecommerce;


# Understanding the Collection Object

The variable **customers** is a **Collection object**.

It represents the **customers** collection inside the **ecommerce** database.

Using this object, we can perform operations such as:

- Insert documents
- Retrieve documents
- Update documents
- Delete documents
- Aggregate data

Since no documents have been inserted yet, the **customers** collection has not been created physically.

In [17]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

[]

In [18]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Document

A **document** is the basic unit of data storage in MongoDB.

A document is a collection of **field-value pairs**, similar to a JSON object.

For example, a customer in an e-commerce application can be represented as a single document.

```json
{
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul@example.com",
    "city": "Hyderabad",
    "membership": "Gold",
    "is_active": true
}
```

Each field stores a specific piece of information about the customer.

Unlike relational databases, MongoDB documents can have different fields. This flexible schema makes MongoDB suitable for applications where the data structure changes frequently.

# JSON vs BSON

Although MongoDB documents look like **JSON**, MongoDB actually stores them internally as **BSON (Binary JSON)**.

BSON extends JSON by supporting additional data types and enabling faster storage and retrieval.

| JSON | BSON |
|------|------|
| Text-based format | Binary format |
| Human-readable | Machine-efficient |
| Limited data types | Supports additional data types such as ObjectId, Date, Binary, Decimal128, etc. |
| Used for data interchange | Used for internal storage by MongoDB |

When we write documents in Python or in the MongoDB Shell, they appear like JSON, but MongoDB automatically converts them into BSON before storing them.

# The `_id` Field and ObjectId

Every document stored in MongoDB must contain a unique field named **`_id`**.

If we do not provide an `_id`, MongoDB automatically generates one.

The automatically generated `_id` is usually of type **ObjectId**.

Example:

```text
ObjectId("6878fd5f5d5d4e8c2f7c3b9a")
```

The `_id` field serves as the **primary key** of a MongoDB document.

Its purpose is to uniquely identify every document within a collection.

> **Note:**  
> During our first insert operation, we will not specify the `_id` field. MongoDB will generate it automatically.

In [19]:
# =====================================
# PyMongo Command
# =====================================

from bson import ObjectId

# Generate a sample ObjectId (for demonstration)

sample_id = ObjectId()

sample_id

ObjectId('6a5cb76683106d9367c89b86')

In [20]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "ObjectId()"

ObjectId('6a5cb79992d204f8f4d7a6c1')


# Insert a Single Document

The **`insert_one()`** method is used to insert a single document into a MongoDB collection.

If the specified database or collection does not already exist, MongoDB automatically creates them when the first document is inserted.

Since we have not yet inserted any data, this operation will:

- Create the **ecommerce** database.
- Create the **customers** collection.
- Insert one customer document.
- Automatically generate an **`_id`** field because we are not providing one.

In [21]:
# =====================================
# PyMongo Command
# =====================================

customer = {
    "customer_id": 1001,
    "name": "Rahul Sharma",
    "gender": "Male",
    "age": 28,
    "email": "rahul.sharma@example.com",
    "phone": "9876543210",
    "city": "Hyderabad",
    "state": "Telangana",
    "membership": "Gold",
    "join_date": "2025-01-15",
    "is_active": True
}

result = customers.insert_one(customer)

print("Inserted Document ID:", result.inserted_id)

Inserted Document ID: 6a5cb7de83106d9367c89b87


In [22]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval '
# use ecommerce
# db.customers.insertOne({
#     customer_id: 1001,
#     name: "Rahul Sharma",
#     gender: "Male",
#     age: 28,
#     email: "rahul.sharma@example.com",
#     phone: "9876543210",
#     city: "Hyderabad",
#     state: "Telangana",
#     membership: "Gold",
#     join_date: "2025-01-15",
#     is_active: true
# })'

# Understanding the Return Value

The `insert_one()` method returns an **InsertOneResult** object.

The most commonly used attribute is:

- **`inserted_id`** – Returns the `_id` of the newly inserted document.

This value can be stored and used later to retrieve, update, or delete the same document.

In [23]:
# =====================================
# PyMongo Command
# =====================================

type(result)

pymongo.results.InsertOneResult

In [24]:
# =====================================
# PyMongo Command
# =====================================

result.inserted_id

ObjectId('6a5cb7de83106d9367c89b87')

# Verify Database and Collection Creation

Earlier, the **ecommerce** database and **customers** collection did not physically exist.

After inserting the first document, MongoDB automatically creates both.

Let's verify this.

In [25]:
# =====================================
# PyMongo Command
# =====================================

client.list_database_names()

['admin', 'config', 'ecommerce', 'local']

In [26]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "show dbs"

admin       40.00 KiB
config     108.00 KiB
ecommerce   40.00 KiB
local       64.00 KiB


In [27]:
# =====================================
# PyMongo Command
# =====================================

db.list_collection_names()

['customers']

In [28]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; show collections"

switched to db ecommerce;


# Retrieve a Single Document

The **`find_one()`** method retrieves the **first document** that matches the specified query.

If no query is provided, MongoDB returns the **first document** in the collection.

The returned value is a Python dictionary (`dict`) representing a MongoDB document.

In [29]:
# =====================================
# PyMongo Command
# =====================================

customers.find_one()

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'customer_id': 1001,
 'name': 'Rahul Sharma',
 'gender': 'Male',
 'age': 28,
 'email': 'rahul.sharma@example.com',
 'phone': '9876543210',
 'city': 'Hyderabad',
 'state': 'Telangana',
 'membership': 'Gold',
 'join_date': '2025-01-15',
 'is_active': True}

In [30]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.findOne()"

switched to db ecommerce;


# Retrieve All Documents

The **`find()`** method retrieves **all documents** that match the specified query.

If no query is provided, every document in the collection is returned.

Unlike `find_one()`, the `find()` method returns a **Cursor object**.

A cursor is an iterator that allows MongoDB to efficiently return documents one at a time instead of loading all documents into memory.

In [31]:
# =====================================
# PyMongo Command
# =====================================

customers.find()

In [38]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "use ecommerce; db.customers.find()"

switched to db ecommerce;


# Display Documents Using a Loop

Since the `find()` method returns a **Cursor**, we usually iterate through it using a loop.

Each iteration returns one document from the collection.

In [34]:
# =====================================
# PyMongo Command
# =====================================

for customer in customers.find():
    print(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'), 'customer_id': 1001, 'name': 'Rahul Sharma', 'gender': 'Male', 'age': 28, 'email': 'rahul.sharma@example.com', 'phone': '9876543210', 'city': 'Hyderabad', 'state': 'Telangana', 'membership': 'Gold', 'join_date': '2025-01-15', 'is_active': True}


In [39]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find()"

[
  {
    _id: ObjectId('6a5cb7de83106d9367c89b87'),
    customer_id: 1001,
    name: 'Rahul Sharma',
    gender: 'Male',
    age: 28,
    email: 'rahul.sharma@example.com',
    phone: '9876543210',
    city: 'Hyderabad',
    state: 'Telangana',
    membership: 'Gold',
    join_date: '2025-01-15',
    is_active: true
  }
]


# Count the Number of Documents

The **`count_documents()`** method returns the number of documents that match a specified filter.

Passing an empty filter `{}` counts every document in the collection.

In [40]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

1

In [41]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

1


# Insert Multiple Documents

The **`insert_many()`** method inserts multiple documents into a collection in a single operation.

This method accepts a **list of dictionaries**, where each dictionary represents one document.

Advantages of using `insert_many()`:

- Inserts multiple documents efficiently.
- Reduces communication with the MongoDB server.
- Returns the `_id` values of all inserted documents.

In this section, we will populate our **customers** collection with additional customer records for future examples.

In [42]:
# =====================================
# PyMongo Command
# =====================================

customers_list = [
    {
        "customer_id": 1002,
        "name": "Priya Reddy",
        "gender": "Female",
        "age": 25,
        "email": "priya.reddy@example.com",
        "phone": "9876543211",
        "city": "Hyderabad",
        "state": "Telangana",
        "membership": "Silver",
        "join_date": "2025-02-10",
        "is_active": True
    },
    {
        "customer_id": 1003,
        "name": "Arjun Kumar",
        "gender": "Male",
        "age": 32,
        "email": "arjun.kumar@example.com",
        "phone": "9876543212",
        "city": "Bengaluru",
        "state": "Karnataka",
        "membership": "Gold",
        "join_date": "2024-12-20",
        "is_active": True
    },
    {
        "customer_id": 1004,
        "name": "Sneha Patel",
        "gender": "Female",
        "age": 29,
        "email": "sneha.patel@example.com",
        "phone": "9876543213",
        "city": "Mumbai",
        "state": "Maharashtra",
        "membership": "Platinum",
        "join_date": "2025-03-05",
        "is_active": True
    },
    {
        "customer_id": 1005,
        "name": "Vikram Singh",
        "gender": "Male",
        "age": 35,
        "email": "vikram.singh@example.com",
        "phone": "9876543214",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Silver",
        "join_date": "2024-11-18",
        "is_active": False
    },
    {
        "customer_id": 1006,
        "name": "Ananya Das",
        "gender": "Female",
        "age": 27,
        "email": "ananya.das@example.com",
        "phone": "9876543215",
        "city": "Kolkata",
        "state": "West Bengal",
        "membership": "Gold",
        "join_date": "2025-01-28",
        "is_active": True
    },
    {
        "customer_id": 1007,
        "name": "Rohit Verma",
        "gender": "Male",
        "age": 31,
        "email": "rohit.verma@example.com",
        "phone": "9876543216",
        "city": "Pune",
        "state": "Maharashtra",
        "membership": "Bronze",
        "join_date": "2025-04-01",
        "is_active": True
    },
    {
        "customer_id": 1008,
        "name": "Kavya Nair",
        "gender": "Female",
        "age": 26,
        "email": "kavya.nair@example.com",
        "phone": "9876543217",
        "city": "Kochi",
        "state": "Kerala",
        "membership": "Silver",
        "join_date": "2025-02-15",
        "is_active": False
    },
    {
        "customer_id": 1009,
        "name": "Amit Joshi",
        "gender": "Male",
        "age": 38,
        "email": "amit.joshi@example.com",
        "phone": "9876543218",
        "city": "Jaipur",
        "state": "Rajasthan",
        "membership": "Gold",
        "join_date": "2024-10-12",
        "is_active": True
    },
    {
        "customer_id": 1010,
        "name": "Neha Sharma",
        "gender": "Female",
        "age": 24,
        "email": "neha.sharma@example.com",
        "phone": "9876543219",
        "city": "Delhi",
        "state": "Delhi",
        "membership": "Bronze",
        "join_date": "2025-05-08",
        "is_active": True
    },
    {
        "customer_id": 1011,
        "name": "Suresh Iyer",
        "gender": "Male",
        "age": 40,
        "email": "suresh.iyer@example.com",
        "phone": "9876543220",
        "city": "Chennai",
        "state": "Tamil Nadu",
        "membership": "Platinum",
        "join_date": "2024-09-25",
        "is_active": True
    }
]

result = customers.insert_many(customers_list)

print("Inserted", len(result.inserted_ids), "documents.")

Inserted 10 documents.


In [43]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

# !mongosh --eval "
# db.getSiblingDB('ecommerce').customers.insertMany([
#     { customer_id:1002, name:'Priya Reddy', gender:'Female', age:25, email:'priya.reddy@example.com', phone:'9876543211', city:'Hyderabad', state:'Telangana', membership:'Silver', join_date:'2025-02-10', is_active:true },
#     { customer_id:1003, name:'Arjun Kumar', gender:'Male', age:32, email:'arjun.kumar@example.com', phone:'9876543212', city:'Bengaluru', state:'Karnataka', membership:'Gold', join_date:'2024-12-20', is_active:true }
#     // Remaining documents omitted for readability.
# ])
# "

# Verify the Inserted Documents

After inserting multiple documents, let's verify the total number of documents available in the **customers** collection.

In [44]:
# =====================================
# PyMongo Command
# =====================================

customers.count_documents({})

11

In [45]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.countDocuments({})"

11


In [55]:
# =====================================
# PyMongo Command
# =====================================

cursor = customers.find({"membership": "Gold"})

for customer in cursor:
    pprint(customer)

{'_id': ObjectId('6a5cb7de83106d9367c89b87'),
 'age': 28,
 'city': 'Hyderabad',
 'customer_id': 1001,
 'email': 'rahul.sharma@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2025-01-15',
 'membership': 'Gold',
 'name': 'Rahul Sharma',
 'phone': '9876543210',
 'state': 'Telangana'}
{'_id': ObjectId('6a5cba4483106d9367c89b89'),
 'age': 32,
 'city': 'Bengaluru',
 'customer_id': 1003,
 'email': 'arjun.kumar@example.com',
 'gender': 'Male',
 'is_active': True,
 'join_date': '2024-12-20',
 'membership': 'Gold',
 'name': 'Arjun Kumar',
 'phone': '9876543212',
 'state': 'Karnataka'}
{'_id': ObjectId('6a5cba4483106d9367c89b8c'),
 'age': 27,
 'city': 'Kolkata',
 'customer_id': 1006,
 'email': 'ananya.das@example.com',
 'gender': 'Female',
 'is_active': True,
 'join_date': '2025-01-28',
 'membership': 'Gold',
 'name': 'Ananya Das',
 'phone': '9876543215',
 'state': 'West Bengal'}
{'_id': ObjectId('6a5cba4483106d9367c89b8f'),
 'age': 38,
 'city': 'Jaipur',
 'customer_id': 1009,

In [56]:
# =====================================
# mongosh Command (Equivalent)
# =====================================

!mongosh --eval "db.getSiblingDB('ecommerce').customers.find({membership:'Gold'}).forEach(doc => printjson(doc))"

{
  _id: ObjectId('6a5cb7de83106d9367c89b87'),
  customer_id: 1001,
  name: 'Rahul Sharma',
  gender: 'Male',
  age: 28,
  email: 'rahul.sharma@example.com',
  phone: '9876543210',
  city: 'Hyderabad',
  state: 'Telangana',
  membership: 'Gold',
  join_date: '2025-01-15',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b89'),
  customer_id: 1003,
  name: 'Arjun Kumar',
  gender: 'Male',
  age: 32,
  email: 'arjun.kumar@example.com',
  phone: '9876543212',
  city: 'Bengaluru',
  state: 'Karnataka',
  membership: 'Gold',
  join_date: '2024-12-20',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8c'),
  customer_id: 1006,
  name: 'Ananya Das',
  gender: 'Female',
  age: 27,
  email: 'ananya.das@example.com',
  phone: '9876543215',
  city: 'Kolkata',
  state: 'West Bengal',
  membership: 'Gold',
  join_date: '2025-01-28',
  is_active: true
}
{
  _id: ObjectId('6a5cba4483106d9367c89b8f'),
  customer_id: 1009,
  name: 'Amit Joshi',
  gender: 'Male',
  age: 38,
  emai